# Copilot Org Data — Direct Ingester (Fabric)

End-to-end Lakehouse loader for **org / HRIS data** (manager, department, location) that calls Microsoft Graph directly from inside Fabric. Replaces the prior flow:

```
OLD:  PowerShell → CSV → Files/org_raw/ → Copilot_Org_Data_Loader.ipynb → Delta
NEW:  This notebook (Graph → Delta)
```

**Source endpoint**: `/v1.0/users?$select=...&$expand=manager($select=userPrincipalName)&$top=999` — pages through every Entra user with org-relevant fields and resolves manager UPN inline.

**Output**: Lakehouse Delta table `dbo.copilot_org_data` (same name + schema as the existing CSV loader, so the PBIT works without changes).

**Permissions**: app registration with `User.Read.All` (Application permission, admin-consented).


## 1. Configuration

In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'
CLIENT_ID     = '<your-app-reg-client-id>'
CLIENT_SECRET = '<your-client-secret-value>'

OUTPUT_TABLE  = 'dbo.copilot_org_data'
WRITE_MODE    = 'overwrite'
ALLOW_EMPTY_SNAPSHOT = False   # set True only for an intentional empty first install


## 2. Authenticate to Microsoft Graph

In [ ]:
import requests

def get_graph_token(tenant_id, client_id, client_secret):
    url  = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token'
    body = {
        'client_id':     client_id,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': client_secret,
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

token   = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
headers = {'Authorization': f'Bearer {token}'}
print('✓ Graph token acquired.')


## 3. Page through `/users` with manager expand

Graph caps `$top` at 999. Follow `@odata.nextLink` until exhausted.


In [ ]:
select_fields  = 'userPrincipalName,displayName,department,jobTitle,companyName,officeLocation,city,country,accountEnabled'
expand_manager = 'manager($select=userPrincipalName)'
next_url = f'https://graph.microsoft.com/v1.0/users?$select={select_fields}&$expand={expand_manager}&$top=999'


def _validate_users_page(data, page_number):
    if not isinstance(data, dict):
        raise ValueError(f'Graph /users page {page_number} did not return an object.')
    if 'value' not in data:
        raise ValueError(f"Graph /users page {page_number} is missing required 'value'; refusing to overwrite org data.")
    value = data['value']
    if not isinstance(value, list):
        raise ValueError(f"Graph /users page {page_number} returned a non-list 'value'; refusing to overwrite org data.")
    next_link = data.get('@odata.nextLink')
    if next_link is not None and (not isinstance(next_link, str) or not next_link.strip()):
        raise ValueError(f"Graph /users page {page_number} returned an invalid '@odata.nextLink'.")
    for item_number, item in enumerate(value, start=1):
        if not isinstance(item, dict):
            raise ValueError(f'Graph /users page {page_number} item {item_number} is not an object.')
    return value, next_link


users = []
seen_urls = set()
page_number = 0
while next_url:
    if next_url in seen_urls:
        raise ValueError(f'Graph /users paging loop detected at page {page_number + 1}.')
    seen_urls.add(next_url)
    page_number += 1
    r = requests.get(next_url, headers=headers, timeout=60)
    r.raise_for_status()
    page, next_url = _validate_users_page(r.json(), page_number)
    users.extend(page)
    print(f'  Fetched {len(users):,} users so far...')

print(f'✓ Total users: {len(users):,}')


## 4. Flatten manager UPN + canonicalise columns

`manager` is a nested object — flatten to `managerUPN`. Then rename to match the existing loader's canonical schema (`PersonId`, `Organization`, `JobTitle`).


In [ ]:
def _normalise_identity(value):
    return (value or '').strip().lower()


def _canonical_org_row(user):
    if not isinstance(user, dict):
        raise ValueError('Org data item is not an object.')
    norm = _normalise_identity(user.get('userPrincipalName'))
    if not norm:
        raise ValueError('Org data row is missing userPrincipalName; refusing to write an anonymous identity.')
    manager = user.get('manager')
    manager_upn = ''
    if manager not in (None, ''):
        if not isinstance(manager, dict):
            raise ValueError(f'Org data row {norm!r} has a non-object manager payload.')
        manager_upn = manager.get('userPrincipalName', '') or ''
    return {
        'PersonId':         user.get('userPrincipalName'),
        'displayName':      user.get('displayName'),
        'Organization':     user.get('department'),
        'JobTitle':         user.get('jobTitle'),
        'companyName':      user.get('companyName'),
        'officeLocation':   user.get('officeLocation'),
        'city':             user.get('city'),
        'country':          user.get('country'),
        'accountEnabled':   str(user.get('accountEnabled', '')),
        'managerUPN':       manager_upn,
    }


def _dedupe_org_rows(rows):
    seen = {}
    deduped = []
    for row in rows:
        norm = _normalise_identity(row.get('PersonId'))
        comparable = {key: '' if value is None else str(value).strip() for key, value in row.items()}
        prior = seen.get(norm)
        if prior is None:
            seen[norm] = comparable
            deduped.append(row)
            continue
        if comparable != prior:
            raise ValueError(f'Conflicting org rows detected for {norm!r}; refusing to overwrite a good snapshot.')
    return deduped


rows = _dedupe_org_rows([_canonical_org_row(user) for user in users])
print(f'Built {len(rows):,} canonicalised rows.')


## 4b. Build full manager hierarchy (recursive chain)

Walks each person's `managerUPN` chain to the top of the org and flattens it into `Level0_Name`..`LevelN_Name` + `OrgLevel` + `HierarchyPath` + `IsManager` + `DirectReports`, mirroring the SharePoint-path (PAX) org builder. Pure in-memory — every user and their direct `managerUPN` are already in `rows`, so **no extra Graph calls** are needed. Cycle-safe.


In [ ]:
# Deepest LevelN column to emit. Mirrors the PAX default; harmless if the org
# is shallower (extra LevelN columns just come back empty).
MAX_ORG_LEVELS = 14

HIER_FIXED   = ['OrgLevel', 'HierarchyPath', 'TopOfChain_Name', 'IsManager', 'DirectReports']
HIER_LEVELS  = [f'Level{i}_Name' for i in range(MAX_ORG_LEVELS + 1)]
HIER_COLUMNS = HIER_FIXED + HIER_LEVELS


def build_hierarchy(rows, max_levels=MAX_ORG_LEVELS):
    """Flatten each person's managerUPN chain into Level0..N + org metadata."""
    def _n(v):
        return (v or '').strip().lower()

    name_of, mgr_of = {}, {}
    for row in rows:
        user = _n(row.get('PersonId'))
        if not user:
            continue
        name_of[user] = (row.get('displayName') or row.get('PersonId') or '').strip()
        mgr_of[user] = _n(row.get('managerUPN'))

    direct = {user: 0 for user in name_of}
    for user, manager in mgr_of.items():
        if manager and manager in direct:
            direct[manager] += 1

    out = {}
    for user in name_of:
        chain, seen_chain, cur = [], set(), user
        while cur and cur in name_of:
            if cur in seen_chain:
                raise ValueError(f'Cycle detected in manager hierarchy at {cur!r}.')
            seen_chain.add(cur)
            chain.append(cur)
            cur = mgr_of.get(cur, '')
        chain = list(reversed(chain))
        rec = {column: '' for column in HIER_COLUMNS}
        for idx in range(max_levels + 1):
            rec[f'Level{idx}_Name'] = name_of.get(chain[idx], '') if idx < len(chain) else ''
        rec['OrgLevel'] = str(len(chain) - 1)
        rec['HierarchyPath'] = ' > '.join(name_of.get(node, node) for node in chain)
        rec['TopOfChain_Name'] = name_of.get(chain[0], '') if chain else ''
        rec['IsManager'] = 'TRUE' if direct.get(user, 0) > 0 else 'FALSE'
        rec['DirectReports'] = str(direct.get(user, 0))
        out[user] = rec
    return out


_hier = build_hierarchy(rows, MAX_ORG_LEVELS)
_blank = {column: '' for column in HIER_COLUMNS}
for row in rows:
    user = (row.get('PersonId') or '').strip().lower()
    row.update(_hier.get(user, _blank))

_mgrs = sum(1 for row in rows if row.get('IsManager') == 'TRUE')
_depth = max((int(row.get('OrgLevel') or 0) for row in rows), default=0)
print(f'Built manager hierarchy for {len(_hier):,} people | managers: {_mgrs:,} | max depth: {_depth}')


## 5. Build Spark DataFrame + add normalised key + sanitise columns

Uses an **explicit schema** because Graph's `/users` response can have all-`None` values for some columns on small tenants (e.g. nobody has a `country` set), and Spark's automatic schema inference fails on those with `[CANNOT_DETERMINE_TYPE]`.


In [ ]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType


def _table_exists(table_name):
    return bool(spark.catalog.tableExists(table_name))


def _guard_org_snapshot(row_count, table_name):
    exists = _table_exists(table_name)
    if row_count == 0 and exists:
        raise ValueError(f'Fetched 0 org rows; refusing to replace existing {table_name}.')
    if row_count == 0 and not ALLOW_EMPTY_SNAPSHOT:
        raise ValueError(
            f'Fetched 0 org rows and {table_name} does not exist yet. '
            'Set ALLOW_EMPTY_SNAPSHOT = True only for an intentional empty first install.'
        )


schema = StructType([
    StructField('PersonId',        StringType(), True),
    StructField('displayName',     StringType(), True),
    StructField('Organization',    StringType(), True),
    StructField('JobTitle',        StringType(), True),
    StructField('companyName',     StringType(), True),
    StructField('officeLocation',  StringType(), True),
    StructField('city',            StringType(), True),
    StructField('country',         StringType(), True),
    StructField('accountEnabled',  StringType(), True),
    StructField('managerUPN',      StringType(), True),
])
for column in HIER_COLUMNS:
    schema = schema.add(StructField(column, StringType(), True))

df = spark.createDataFrame(rows, schema=schema)
df = df.withColumn(
    'PersonId_Normalized',
    F.when(F.col('PersonId').isNull(), None)
     .otherwise(F.lower(F.trim(F.col('PersonId').cast('string'))))
)

duplicate_people = (
    df.filter(F.col('PersonId_Normalized').isNotNull() & (F.col('PersonId_Normalized') != ''))
      .groupBy('PersonId_Normalized').count()
      .filter(F.col('count') > 1)
      .count()
)
if duplicate_people:
    raise ValueError('Org data still contains duplicate PersonId_Normalized values after validation.')

total = df.count()
_guard_org_snapshot(total, OUTPUT_TABLE)
df = df.withColumn('TotalEmployees', F.lit(str(total)))

_INVALID = re.compile(r'[ ,;{}()\n\t=]')
df = df.toDF(*[_INVALID.sub('_', c) for c in df.columns])

print(f'Final rows: {total:,}')
print('Columns:', df.columns)


## 6. Write to Lakehouse Delta table

In [ ]:
(df.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows written to {OUTPUT_TABLE}: {row_count:,}')


## 7. Verify

In [ ]:
tbl = spark.table(OUTPUT_TABLE)
tbl.select('PersonId', 'PersonId_Normalized', 'Organization', 'JobTitle', 'managerUPN').show(10, truncate=False)
tbl.groupBy('Organization').count().orderBy(F.desc('count')).show(20, truncate=False)
tbl.select('PersonId', 'displayName', 'managerUPN', 'OrgLevel', 'IsManager', 'DirectReports', 'HierarchyPath').show(10, truncate=False)


---
**Connect the PBIT**: this table is consumed by the `Chat + Agent Org Data` query in both AI-in-One and AI Business Value dashboards. Once this notebook has run, leave the `Org Data File` parameter blank when opening the PBIT — refresh sources from `dbo.copilot_org_data` directly via the Fabric SQL endpoint.
